In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [ ]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
X, y = make_friedman1(n_samples=10000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=1000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [12]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.SGD(model.parameters(), lr=2e-2)

all_loss = {}
for epoch in range(1000):
    print("epoch: ", epoch, end="")
    all_loss[epoch + 1] = 0
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step()

        all_loss[epoch + 1] += loss
    all_loss[epoch + 1] /= len(data_loader)
    print(", loss: {}".format(all_loss[epoch + 1].cpu().detach().numpy().item()))

epoch:  0, loss: 0.6445945501327515
epoch:  1, loss: 0.586988627910614
epoch:  2, loss: 0.5378702282905579
epoch:  3, loss: 0.49405142664909363
epoch:  4, loss: 0.4541774094104767
epoch:  5, loss: 0.41784119606018066
epoch:  6, loss: 0.38470590114593506
epoch:  7, loss: 0.3544743061065674
epoch:  8, loss: 0.32741937041282654
epoch:  9, loss: 0.30456212162971497
epoch:  10, loss: 0.2835778295993805
epoch:  11, loss: 0.26424044370651245
epoch:  12, loss: 0.24641989171504974
epoch:  13, loss: 0.22999677062034607
epoch:  14, loss: 0.21486172080039978
epoch:  15, loss: 0.20091375708580017
epoch:  16, loss: 0.18805953860282898
epoch:  17, loss: 0.17621348798274994
epoch:  18, loss: 0.1652964949607849
epoch:  19, loss: 0.15523558855056763
epoch:  20, loss: 0.14596356451511383
epoch:  21, loss: 0.13741852343082428
epoch:  22, loss: 0.12954357266426086
epoch:  23, loss: 0.12228607386350632
epoch:  24, loss: 0.11559758335351944
epoch:  25, loss: 0.10943351686000824
epoch:  26, loss: 0.1037527099

In [13]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = -63080072.28265376
Test metrics:  R2 = 0.0


In [14]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
opt = optim.Adam(model.parameters(), lr=1e-2)

all_loss = {}
for epoch in range(1000):
    print("epoch: ", epoch, end="")
    all_loss[epoch + 1] = 0
    for batch_idx, (b_x, b_y) in enumerate(data_loader):
        pre = model(b_x)
        loss = loss_fn(pre, b_y)
        opt.zero_grad()
        loss.backward()

        # parameter update step based on optimizer
        opt.step()

        all_loss[epoch + 1] += loss
    all_loss[epoch + 1] /= len(data_loader)
    print(", loss: {}".format(all_loss[epoch + 1].cpu().detach().cpu().numpy().item()))

epoch:  0, loss: 0.31839901208877563
epoch:  1, loss: 0.2543609142303467
epoch:  2, loss: 0.20651645958423615
epoch:  3, loss: 0.16247543692588806
epoch:  4, loss: 0.11504729092121124
epoch:  5, loss: 0.07132381200790405
epoch:  6, loss: 0.03943996503949165
epoch:  7, loss: 0.03169899061322212
epoch:  8, loss: 0.05611386150121689
epoch:  9, loss: 0.0735391229391098
epoch:  10, loss: 0.06686240434646606
epoch:  11, loss: 0.05067741498351097
epoch:  12, loss: 0.037584803998470306
epoch:  13, loss: 0.03152133524417877
epoch:  14, loss: 0.03116188943386078
epoch:  15, loss: 0.03378456458449364
epoch:  16, loss: 0.03715807944536209
epoch:  17, loss: 0.03992330655455589
epoch:  18, loss: 0.041456133127212524
epoch:  19, loss: 0.04161087051033974
epoch:  20, loss: 0.04054694250226021
epoch:  21, loss: 0.03860392048954964
epoch:  22, loss: 0.03622474893927574
epoch:  23, loss: 0.03389284759759903
epoch:  24, loss: 0.03206024318933487
epoch:  25, loss: 0.031058287248015404
epoch:  26, loss: 0.0

In [15]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9011507362318683
Test metrics:  R2 = 0.853575775358627
